# Crop Disease Detection — Training on Updated Dataset (41 Classes)
EfficientNetB3 with stratified split, advanced augmentation, label smoothing, and per-class metrics.

**Kaggle: Add Data → Upload → select `FYP-dataset-updated.zip`**
**Runtime → Accelerator → GPU T4 x2**

## Key improvements:
- **41 classes** — updated dataset with Grape, Pepper, Soybean, and expanded Tomato classes
- **Stratified 70/15/15 split** — preserves rare class distribution
- **Real non-plant Unknown class** — downloads actual non-leaf images (animals, buildings, objects)
- **Stronger augmentation** — brightness, contrast, rotation, zoom, shear
- **Dropout 0.5 + L2 regularization** — better generalization
- **Label smoothing (0.1)** — prevents overconfidence
- **Cosine decay LR** — better fine-tuning schedule
- **Per-class metrics** — identify which diseases perform poorly
- **Gaussian noise + leaf-like Unknown textures** — teaches model to reject random patterns and unfamiliar leaf shapes
- **Model checkpoint** — saves best weights, not last

## 1. Imports


In [ ]:
import os, json, shutil, random, zipfile, math
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.optimizers.schedules import CosineDecay
from PIL import Image
from sklearn.model_selection import train_test_split

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Dataset Path (Kaggle)
Add `FYP-dataset-updated.zip` as a Kaggle Dataset via **Add Data → Upload**.
Kaggle automatically extracts it to `/kaggle/input/`. Verify the path below.

In [ ]:
# Detect Kaggle dataset path and copy to writable working directory
KAGGLE_INPUT = '/kaggle/input'
WORK_DIR = 'FYP-dataset-updated'

if os.path.exists(KAGGLE_INPUT):
    items = os.listdir(KAGGLE_INPUT)
    print('Kaggle input dirs:', items)
    if items:
        dataset_dir = os.path.join(KAGGLE_INPUT, items[0])
        subdirs = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
        print('Subdirectories:', subdirs)
        if 'FYP-dataset-updated' in subdirs:
            src_dir = os.path.join(dataset_dir, 'FYP-dataset-updated')
        elif subdirs:
            src_dir = os.path.join(dataset_dir, subdirs[0])
        else:
            src_dir = dataset_dir
        print(f'Source dataset: {src_dir}')

        # Copy to writable working directory (Kaggle input is read-only)
        if not os.path.exists(WORK_DIR):
            print(f'Copying dataset to {WORK_DIR}...')
            shutil.copytree(src_dir, WORK_DIR)
            print('Copy complete')
        else:
            print(f'{WORK_DIR} already exists')
    else:
        print('No dataset found in /kaggle/input/')
else:
    print('Not running on Kaggle. Place FYP-dataset-updated/ in the working directory.')

DATASET_DIR = WORK_DIR
print(f'DATASET_DIR = {DATASET_DIR}')

## 3. Download Real Non-Plant Images for Unknown Class
Downloads diverse images (animals, buildings, vehicles, objects, food) so the model learns to reject real-world non-leaf photos instead of fake noise patterns.

In [ ]:
import requests
OUT_DIR = 'FYP-dataset-updated/Unknown___Unknown'
os.makedirs(OUT_DIR, exist_ok=True)

existing = [f for f in os.listdir(OUT_DIR) if f.endswith(('.jpg','.jpeg','.png'))]
if len(existing) < 200:
    categories = [
        'animal', 'architecture', 'car', 'city', 'dog', 'food', 'furniture',
        'interior', 'mountain', 'ocean', 'people', 'road', 'skyline', 'street',
        'texture', 'tool', 'train', 'beach', 'sunset', 'cat', 'bicycle', 'bridge'
    ]
    downloaded = 0
    errors = 0
    for cat in categories:
        if downloaded >= 500:
            break
        for attempt in range(30):
            if downloaded >= 500:
                break
            try:
                url = f'https://source.unsplash.com/random/300x300/?{cat}&sig={downloaded}'
                r = requests.get(url, timeout=10, allow_redirects=True)
                if r.status_code == 200 and len(r.content) > 1000:
                    fname = f'{cat}_{downloaded:04d}.jpg'
                    with open(os.path.join(OUT_DIR, fname), 'wb') as f:
                        f.write(r.content)
                    downloaded += 1
                    if downloaded % 50 == 0:
                        print(f'Downloaded {downloaded}/500...')
            except Exception as e:
                errors += 1
                if errors > 100:
                    break
    print(f'Downloaded {downloaded} real non-plant images')
    if len(os.listdir(OUT_DIR)) < 50:
        print('WARNING: Too few images. Falling back to synthetic.')
        rng = random.Random(42)
        while len([f for f in os.listdir(OUT_DIR) if f.endswith(('.jpg','.jpeg','.png'))]) < 300:
            arr = np.random.randint(0, 256, (300, 300, 3), dtype=np.uint8)
            Image.fromarray(arr).save(os.path.join(OUT_DIR, f'synth_{rng.randint(0,99999):05d}.jpg'))
        print('Fallback synthetic images generated')
else:
    print(f'Unknown images already exist ({len(existing)} files)')

## 4. Generate Synthetic Unknown Images (Gaussian Noise + Leaf-Like Textures)
Gaussian noise at 5 intensity levels teaches the model that random patterns → Unknown.
Leaf-like green/brown blobs teach the model that unfamiliar leaf-looking textures → Unknown.
This prevents the model from confidently forcing a random or unseen-leaf image into one of the 41 known classes.

In [ ]:
from PIL import ImageDraw

OUT_DIR = 'FYP-dataset-updated/Unknown___Unknown'
rng = random.Random(42)

noise_count = len([f for f in os.listdir(OUT_DIR) if f.startswith('gn_')])
texture_count = len([f for f in os.listdir(OUT_DIR) if f.startswith('lt_')])

if noise_count < 150:
    for sigma in [8, 20, 40, 80, 160]:
        for i in range(40):
            arr = np.clip(np.random.randn(300, 300, 3) * sigma + 128, 0, 255).astype(np.uint8)
            Image.fromarray(arr).save(os.path.join(OUT_DIR, f'gn_{sigma}_{i:04d}.jpg'))
    print(f'Generated Gaussian noise images (5 levels, 200 total)')

if texture_count < 150:
    for i in range(300):
        bg = rng.randint(30, 90)
        img = Image.new('RGB', (300, 300), (bg, bg + 20, bg))
        draw = ImageDraw.Draw(img)
        for _ in range(rng.randint(3, 12)):
            cx, cy = rng.randint(30, 270), rng.randint(30, 270)
            rx, ry = rng.randint(15, 100), rng.randint(10, 60)
            g = rng.randint(60, 200)
            r = rng.randint(20, min(g, 110))
            b = rng.randint(15, 55)
            draw.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=(r, g, b))
        for _ in range(rng.randint(0, 4)):
            x1, y1, x2, y2 = [rng.randint(0, 300) for _ in range(4)]
            draw.line([(x1, y1), (x2, y2)], fill=(rng.randint(40, 80), rng.randint(60, 120), rng.randint(20, 50)), width=rng.randint(1, 3))
        img.save(os.path.join(OUT_DIR, f'lt_{i:04d}.jpg'))
    print(f'Generated leaf-like texture images (300 total)')

total_unknown = len([f for f in os.listdir(OUT_DIR) if f.endswith(('.jpg','.jpeg','.png'))])
print(f'Total Unknown___Unknown images: {total_unknown}')

## 5. Training Configuration


In [ ]:
if 'DATASET_DIR' not in dir() or not os.path.exists(str(DATASET_DIR)):
    DATASET_DIR = 'FYP-dataset-updated'
IMG_SIZE = (300, 300)
BATCH_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
EPOCHS_PHASE1 = 20
EPOCHS_PHASE2 = 30
LR_PHASE1 = 5e-4
LR_PHASE2 = 1e-4
DROPOUT_RATE = 0.5
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 1e-4
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

## 6. Stratified Train / Validation / Test Split (70 / 15 / 15)
Splits images per class to preserve rare-class distribution in all three sets.

In [ ]:
base_dir = 'data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

classes = sorted(os.listdir(DATASET_DIR))

for cls in classes:
    src = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(src):
        continue
    images = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * 0.70)
    n_val = int(n_total * 0.15)
    train_imgs = images[:n_train]
    val_imgs = images[n_train:n_train + n_val]
    test_imgs = images[n_train + n_val:]

    for split_dir, split_imgs in [
        (os.path.join(train_dir, cls), train_imgs),
        (os.path.join(val_dir, cls), val_imgs),
        (os.path.join(test_dir, cls), test_imgs),
    ]:
        os.makedirs(split_dir, exist_ok=True)
        for img in split_imgs:
            shutil.copy2(os.path.join(src, img), os.path.join(split_dir, img))

    print(f'{cls:45s} {len(train_imgs):4d} train, {len(val_imgs):4d} val, {len(test_imgs):4d} test')
print(f'\nTotal classes: {len(classes)}')

## 7. Compute Class Weights
Inverse-frequency weighting so the model pays more attention to rare classes.

In [ ]:
class_counts = {}
for i, cls in enumerate(classes):
    cls_dir = os.path.join(train_dir, cls)
    n = len([f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if n > 0:
        class_counts[i] = n

if not class_counts:
    raise RuntimeError('No training images found in any class! Check dataset path.')

max_count = max(class_counts.values())
class_weight = {i: max_count / n for i, n in class_counts.items()}

print('Class weights:')
for i, cls in enumerate(classes):
    n = class_counts.get(i, 0)
    if n == 0:
        print(f'  {cls:45s} {"SKIPPED":>4s} images  (no training samples)')
    else:
        bar = '#' * min(50, int(class_weight[i]))
        print(f'  {cls:45s} {n:4d} images  weight={class_weight[i]:6.2f}  {bar}')

## 8. Data Augmentation
Strong augmentation for training (rotation, shift, zoom, brightness, shear, flip).
No augmentation for validation / test.

In [ ]:
def get_train_datagen():
    return tf.keras.preprocessing.image.ImageDataGenerator(
        rotation_range=40,
        width_shift_range=0.25,
        height_shift_range=0.25,
        shear_range=0.2,
        zoom_range=0.3,
        brightness_range=(0.6, 1.4),
        horizontal_flip=True,
        vertical_flip=False,
        fill_mode='reflect',
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    )

def get_eval_datagen():
    return tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    )

## 9. Create Data Generators


In [ ]:
train_gen = get_train_datagen().flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)
val_gen = get_eval_datagen().flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_gen = get_eval_datagen().flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'\nTraining samples: {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Test samples: {test_gen.samples}')

## 10. Build Model (EfficientNetB3)
Pretrained ImageNet weights, global pooling, dropout 0.5, L2 regularization, 41-class softmax output.

In [ ]:
base_model = EfficientNetB3(
    include_top=False, weights='imagenet',
    input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(512, activation='relu',
                 kernel_regularizer=regularizers.l2(WEIGHT_DECAY))(x)
x = layers.Dropout(DROPOUT_RATE)(x)
outputs = layers.Dense(
    len(classes), activation='softmax',
    kernel_regularizer=regularizers.l2(WEIGHT_DECAY)
)(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

## 11. Phase 1 — Train Top Layers Only
Frozen backbone, Adam 5e-4, label smoothing 0.1, early stopping, checkpoint.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_PHASE1),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=7, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model_phase1.weights.h5',
        monitor='val_accuracy', save_best_only=True,
        save_weights_only=True
    ),
]

print('Phase 1: Training top layers...')
history1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_PHASE1, callbacks=callbacks,
    class_weight=class_weight, verbose=1
)

## 12. Phase 2 — Fine-Tune Entire Model
Unfreeze backbone (freeze first 100 layers), cosine decay LR 1e-4 → 1e-6, label smoothing.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

total_steps = len(train_gen) * EPOCHS_PHASE2
cosine_schedule = CosineDecay(
    initial_learning_rate=LR_PHASE2,
    decay_steps=total_steps,
    alpha=0.01
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(cosine_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

callbacks2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=7, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model_phase2.weights.h5',
        monitor='val_accuracy', save_best_only=True,
        save_weights_only=True
    ),
]

print('Phase 2: Fine-tuning entire model...')
history2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_PHASE2, callbacks=callbacks2,
    class_weight=class_weight, verbose=1
)

model.load_weights('best_model_phase2.weights.h5')

## 13. Evaluate on Test Set — Overall Metrics


In [ ]:
def clean_name(label):
    name = label.replace('___', ' ').replace('__', ' ').replace('_', ' ')
    return ' '.join(name.split())

test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f'Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test loss: {test_loss:.4f}')

all_preds = model.predict(test_gen, verbose=0)
pred_classes = np.argmax(all_preds, axis=1)
true_classes = test_gen.classes

## 14. Per-Class Metrics
Precision, recall, F1-score for every disease class — identifies which classes need improvement.

In [ ]:
from sklearn.metrics import classification_report

target_names = [clean_name(c) for c in classes]
print(classification_report(true_classes, pred_classes, target_names=target_names, digits=3))

## 15. Save Model Weights & Class Names


In [ ]:
os.makedirs('model_output', exist_ok=True)

model.save_weights('model_output/model.weights.h5')
print('Weights saved to model_output/model.weights.h5')

with open('model_output/class_names.json', 'w') as f:
    json.dump(classes, f, indent=2)
print('Class names saved to model_output/class_names.json')

## 16. Confusion Matrix Plot
Visual heatmap of correct vs misclassified predictions per class.

In [ ]:
from sklearn.metrics import confusion_matrix

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    cm = confusion_matrix(true_classes, pred_classes)
    plt.figure(figsize=(20, 16))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title('Confusion Matrix - Test Set')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('model_output/confusion_matrix.png', dpi=150)
    plt.show()
    print('Confusion matrix saved to model_output/confusion_matrix.png')
except ImportError:
    print('matplotlib/seaborn not available, skipping confusion matrix plot')

## Download Output
Output files are in `model_output/`. From Kaggle:
1. Click the **Data** tab on the right
2. Click **Upload to Kaggle Datasets** or use **Download All**
3. Or click the **Output** tab → **Download** button

Copy these to your FastAPI backend:
- `model.weights.h5` → `backend/updated-model/model.weights.h5`
- `class_names.json` → `backend/class_names.json`


In [ ]:
print('Output files:')
for f in os.listdir('model_output'):
    size = os.path.getsize(os.path.join('model_output', f))
    print(f'  {f:40s} {size / 1024 / 1024:.2f} MB')
print(f'\nKaggle working dir: {os.getcwd()}')